# Tự Động Huấn Luyện & So Sánh Nhiều Cấu Hình (Chạy Được Cả Local & Kaggle)
Notebook này được tối ưu sẵn để **chạy trực tiếp trên Kaggle GPU (T4 / P100)** hoặc trên máy cá nhân:
1. **Tự động nhận diện môi trường:** Tự động tìm đường dẫn dataset GoPro trong `/kaggle/input` hoặc thư mục local.
2. **Danh sách cấu hình (`CONFIG_LIST`):** Tự động huấn luyện lần lượt từng cấu hình qua vòng lặp `for`.
3. **Lưu trữ riêng biệt:** Mỗi cấu hình có file checkpoint `.pth` riêng.
4. **Bảng xếp hạng (Leaderboard):** Tự động xuất bảng so sánh PSNR, SSIM, Loss của từng cấu hình.
5. **Biểu đồ đa đường:** Vẽ các đường cong hội tụ đè lên nhau để so sánh trực quan.
6. **Visual Demo:** Render ảnh so sánh 3 cột: [Ảnh Mờ] vs [Ảnh AI Phục Hồi] vs [Ảnh Gốc Nét].

In [1]:
import sys
import os
import time
import random
import json
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Tạo thư mục weights riêng để lưu file .pth cho gọn gàng
os.makedirs("weights", exist_ok=True)

candidate_paths = [
    '.', '..', 
    '/kaggle/working', 
    '/kaggle/working/Image-processing-project-integrating-MoE-and-Q-Former'
]

for p in candidate_paths:
    if os.path.exists(os.path.join(p, 'models')) and os.path.exists(os.path.join(p, 'utils')):
        abs_p = os.path.abspath(p)
        if abs_p not in sys.path:
            sys.path.insert(0, abs_p)
        break

from models.moe_qformer_net import NAFNetBaseline, MoEQFormerDeblurNet
from utils.dataset import GoProDeblurDataset
from utils.metrics import CharbonnierLoss, calc_psnr, calc_ssim, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {device}")

Thiết bị đang sử dụng: cuda


In [2]:
pretrained_url = "https://huggingface.co/Jacid23/third-eye-models/resolve/5c2e322323888d8d26852f06715ccc6eb61ea795/NAFNet-GoPro-width64.pth"
nafnet_ckpt_path = os.path.join("weights", "NAFNet-GoPro-width64.pth")

if not os.path.exists(nafnet_ckpt_path):
    print("--> Đang tải pretrained NAFNet từ Hugging Face vào thư mục weights/...")
    urllib.request.urlretrieve(pretrained_url, nafnet_ckpt_path)
    print("--> Tải xuống checkpoint thành công!")
else:
    print(f"--> Đã tìm thấy checkpoint tại: {nafnet_ckpt_path}")

nafnet_state_dict = torch.load(nafnet_ckpt_path, map_location=device)
if 'params' in nafnet_state_dict:
    nafnet_state_dict = nafnet_state_dict['params']
elif 'model_state_dict' in nafnet_state_dict:
    nafnet_state_dict = nafnet_state_dict['model_state_dict']

print("--> Đã nạp thành công nafnet_state_dict vào bộ nhớ!")

--> Đã tìm thấy checkpoint tại: weights\NAFNet-GoPro-width64.pth
--> Đã nạp thành công nafnet_state_dict vào bộ nhớ!


### Định nghĩa Hàm Đánh Giá Chung (Evaluation Function)

In [8]:
# Hàm đánh giá PSNR & SSIM trên tập Test
def evaluate_model(model, loader, max_eval_samples=30):
    model.eval()
    psnr_list, ssim_list = [], []
    count = 0
    
    # Thêm tqdm để nhìn thấy tiến trình chạy validation
    pbar_eval = tqdm(loader, desc="[Validating]", leave=False)
    with torch.no_grad():
        for inputs, targets in pbar_eval:
            inputs, targets = inputs.to(device), targets.to(device)
            preds, _ = model(inputs)
            preds = torch.clamp(preds, 0.0, 1.0)
            
            psnr_list.append(calc_psnr(preds, targets))
            ssim_list.append(calc_ssim(preds, targets))
            count += 1
            if max_eval_samples and count >= max_eval_samples:
                break
                
    avg_psnr = sum(psnr_list) / len(psnr_list)
    avg_ssim = sum(ssim_list) / len(ssim_list)
    return avg_psnr, avg_ssim

### Kết quả mô hình baseline NAFNet gốc

In [4]:
# Khởi tạo Dataset dùng chung để đánh giá baseline
def auto_detect_gopro():
    for prefix in ['..', '.']:
        t_train = os.path.join(prefix, 'gopro-data', 'train')
        t_test = os.path.join(prefix, 'gopro-data', 'test')
        if os.path.exists(t_train):
            return t_train, t_test
    return '../gopro-data/train', '../gopro-data/test'

gopro_train_dir, gopro_test_dir = auto_detect_gopro()
test_ds = GoProDeblurDataset(gopro_test_dir, is_train=False)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2)

# 1. Khởi tạo mô hình NAFNet Baseline chuẩn U-Net với cấu hình GoPro
baseline_model = NAFNetBaseline(
    img_channel=3, 
    width=64, 
    enc_blk_nums=[1, 1, 1, 28], 
    middle_blk_num=1, 
    dec_blk_nums=[1, 1, 1, 1]
).to(device)

# 2. Nạp trọng số với strict=True (Khớp hoàn hảo 100% tuyệt đối)
missing_keys, unexpected_keys = baseline_model.load_state_dict(nafnet_state_dict, strict=True)
print(f"--> Khớp trọng số Baseline: Missing = {len(missing_keys)}, Unexpected = {len(unexpected_keys)}")

# 3. Đánh giá trên toàn bộ 1,111 ảnh test
print("\n==================================================")
print(" ĐANG ĐÁNH GIÁ PURE NAFNet BASELINE (CHUẨN PAPER)...")
print("===================================================")

baseline_psnr, baseline_ssim = evaluate_model(baseline_model, test_loader, max_eval_samples=None)

print(f"\n KẾT QUẢ PURE NAFNET BASELINE:")
print(f"   + PSNR: {baseline_psnr:.2f} dB")
print(f"   + SSIM: {baseline_ssim:.4f}")

# Lưu vào từ điển kết quả chung
SCREENING_RESULTS = {}
SCREENING_RESULTS['0_Pure_NAFNet_Baseline'] = {
    'config': {'name': 'Pure_NAFNet', 'lr': 0, 'batch_size': 4, 'freeze_backbone': True, 'description': 'NAFNet gốc thuần túy (Baseline Paper)'},
    'history': {'train_loss': [0], 'val_psnr': [baseline_psnr], 'val_ssim': [baseline_ssim]},
    'best_psnr': baseline_psnr,
    'best_ssim': baseline_ssim,
    'checkpoint': 'NAFNet-GoPro-width64.pth',
    'time_mins': 0.0
}

--> Khớp trọng số Baseline: Missing = 0, Unexpected = 0

 ĐANG ĐÁNH GIÁ PURE NAFNet BASELINE (CHUẨN PAPER)...

 KẾT QUẢ PURE NAFNET BASELINE:
   + PSNR: 32.95 dB
   + SSIM: 0.9430


### Bảng Cấu Hình & Tự Động Định Vị Dataset (`CONFIG_LIST`)

In [9]:
COMMON_CONFIG = {
    'train_dir': gopro_train_dir,
    'test_dir': gopro_test_dir,
    'crop_size': 256, 
    'aux_weight': 0.01,
    'width': 64, 
    'enc_blk_nums': [1, 1, 1, 28], 
    'middle_blk_num': 1, 
    'dec_blk_nums': [1, 1, 1, 1],
    'num_queries': 16, 
    'num_experts': 3,
    'max_eval_samples': 30
}

In [10]:
# Khởi tạo Dataset dùng chung cho các thí nghiệm huấn luyện
train_ds = GoProDeblurDataset(COMMON_CONFIG['train_dir'], crop_size=COMMON_CONFIG['crop_size'], is_train=True)
print(f" Đã nạp thành công: {len(train_ds):,} ảnh Train | {len(test_ds):,} ảnh Test")

 Đã nạp thành công: 2,103 ảnh Train | 1,111 ảnh Test


### Vòng Lặp Huấn Luyện Tự Động Qua Từng Cấu Hình (`for cfg in CONFIG_LIST`)

In [11]:
SCREENING_EPOCHS = 5

CONFIG_LIST = [
    {'name': 'Exp1_LR1e-4_Freeze', 'lr': 1e-4, 'batch_size': 8, 'freeze_backbone': True, 'description': 'Đóng băng backbone, LR nhỏ 1e-4'},
    {'name': 'Exp2_LR2e-4_Freeze', 'lr': 2e-4, 'batch_size': 8, 'freeze_backbone': True, 'description': 'Đóng băng backbone, tăng LR lên 2e-4'},
    {'name': 'Exp3_LR5e-5_Unfreeze', 'lr': 5e-5, 'batch_size': 8, 'freeze_backbone': False, 'description': 'Mở khóa backbone với LR nhỏ 5e-5'}
]

criterion = CharbonnierLoss()

for exp_idx, cfg in enumerate(CONFIG_LIST):
    exp_name = cfg['name']
    print(f"\n================ CHẠY SÀNG LỌC [{exp_idx+1}/3]: {exp_name} ================")
    
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    
    model = MoEQFormerDeblurNet(
        img_channel=3, 
        out_channels=3,
        width=COMMON_CONFIG['width'],
        enc_blk_nums=COMMON_CONFIG['enc_blk_nums'],
        middle_blk_num=COMMON_CONFIG['middle_blk_num'],
        dec_blk_nums=COMMON_CONFIG['dec_blk_nums'],
        num_queries=COMMON_CONFIG['num_queries'], 
        num_experts=COMMON_CONFIG['num_experts']
    ).to(device)
    
    model.load_state_dict(nafnet_state_dict, strict=False)
    if cfg['freeze_backbone']:
        model.freeze_backbone(True)
        
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=cfg['lr'], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SCREENING_EPOCHS, eta_min=1e-6)
    
    history = {'train_loss': [], 'val_psnr': [], 'val_ssim': []}
    best_psnr = baseline_psnr
    best_ssim = baseline_ssim
    temp_save_path = os.path.join("weights", f"temp_screen_{exp_name}.pth")
    
    start_time = time.time()
    for epoch in range(1, SCREENING_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"[{exp_name}] Epoch {epoch:02d}/{SCREENING_EPOCHS:02d}")
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            preds, aux_loss = model(inputs)
            recon_loss = criterion(preds, targets)
            total_loss = recon_loss + COMMON_CONFIG['aux_weight'] * aux_loss
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            running_loss += total_loss.item()
            pbar.set_postfix({'loss': f"{total_loss.item():.4f}"})
            
        epoch_loss = running_loss / len(train_loader)
        scheduler.step()
        
        val_psnr, val_ssim = evaluate_model(model, test_loader, max_eval_samples=COMMON_CONFIG['max_eval_samples'])
        history['train_loss'].append(epoch_loss)
        history['val_psnr'].append(val_psnr)
        history['val_ssim'].append(val_ssim)
        print(f"--> [{exp_name} | Epoch {epoch:02d}] Loss: {epoch_loss:.4f} | PSNR: {val_psnr:.2f} dB")
        
        if val_psnr > best_psnr:
            best_psnr = val_psnr
            best_ssim = val_ssim
            torch.save(model.state_dict(), temp_save_path)
            
    elapsed_time = (time.time() - start_time) / 60.0
    SCREENING_RESULTS[exp_name] = {
        'config': cfg, 'history': history, 'best_psnr': best_psnr, 'best_ssim': best_ssim,
        'checkpoint': temp_save_path, 'time_mins': elapsed_time
    }

best_config_name = max(SCREENING_RESULTS, key=lambda k: SCREENING_RESULTS[k]['best_psnr'])
print(f"\n--> Config tối ưu được chọn: [{best_config_name}]")


================ CHẠY SÀNG LỌC [1/3]: Exp1_LR1e-4_Freeze ================
--> Đã ĐÓNG BĂNG (Freeze) toàn bộ U-Net Backbone!


[Exp1_LR1e-4_Freeze] Epoch 01/05:   0%|          | 0/263 [00:11<?, ?it/s]

[Validating]:   0%|          | 0/278 [00:09<?, ?it/s]

--> [Exp1_LR1e-4_Freeze | Epoch 01] Loss: 0.0208 | PSNR: 34.99 dB


[Exp1_LR1e-4_Freeze] Epoch 02/05:   0%|          | 0/263 [00:10<?, ?it/s]

...

[Exp1_LR1e-4_Freeze] Epoch 03/05:   0%|          | 0/263 [00:09<?, ?it/s]

[Validating]:   0%|          | 0/278 [00:09<?, ?it/s]

--> [Exp1_LR1e-4_Freeze | Epoch 03] Loss: 0.0208 | PSNR: 34.99 dB


[Exp1_LR1e-4_Freeze] Epoch 04/05:   0%|          | 0/263 [00:09<?, ?it/s]

AcceleratorError: Caught AcceleratorError in pin memory thread for device 0.
Original Traceback (most recent call last):
  File "C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\_utils\pin_memory.py", line 34, in do_one_step
    data = pin_memory(data, device)
  File "C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\_utils\pin_memory.py", line 100, in pin_memory
    clone[i] = pin_memory(item, device)
               ~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\admin\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\_utils\pin_memory.py", line 57, in pin_memory
    return data.pin_memory()
           ~~~~~~~~~~~~~~~^^
torch.AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.



### Bảng Xếp Hạng So Sánh Kết Quả (Leaderboard)

In [ ]:
# Tạo bảng tổng hợp so sánh bằng Pandas DataFrame sử dụng SCREENING_RESULTS
summary_data = []
for name, res in SCREENING_RESULTS.items():
    cfg = res['config']
    summary_data.append({
        'Tên Thí Nghiệm': name,
        'LR': cfg['lr'],
        'Batch Size': cfg['batch_size'],
        'Freeze': cfg['freeze_backbone'],
        'Best PSNR (dB)': round(res['best_psnr'], 2),
        'Best SSIM': round(res['best_ssim'], 4),
        'Final Loss': round(res['history']['train_loss'][-1], 4),
        'Thời Gian (phút)': round(res['time_mins'], 1),
        'Mô Tả': cfg['description']
    })

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values(by='Best PSNR (dB)', ascending=False).reset_index(drop=True)
df_summary.index += 1
df_summary.index.name = 'Hạng'

print(" BẢNG SO SÁNH HIỆU NĂNG CÁC CẤU HÌNH:")
display(df_summary)

best_exp_name = df_summary.iloc[0]['Tên Thí Nghiệm']
print(f"\n CẤU HÌNH TỐT NHẤT LÀ: [{best_exp_name}] với PSNR = {df_summary.iloc[0]['Best PSNR (dB)']} dB!")

### Biểu Đồ So Sánh Tiến Trình Hội Tụ Của Các Cấu Hình (Multi-Run Curves)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# 1. So sánh Loss
for i, (name, res) in enumerate(SCREENING_RESULTS.items()):
    c = colors[i % len(colors)]
    axes[0].plot(range(1, len(res['history']['train_loss']) + 1), res['history']['train_loss'], 
                 label=name, color=c, marker='o', linewidth=2)
axes[0].set_title('So Sánh Training Loss (Càng thấp càng tốt)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# 2. So sánh PSNR
for i, (name, res) in enumerate(SCREENING_RESULTS.items()):
    c = colors[i % len(colors)]
    axes[1].plot(range(1, len(res['history']['val_psnr']) + 1), res['history']['val_psnr'], 
                 label=name, color=c, marker='s', linewidth=2)
axes[1].set_title('So Sánh Validation PSNR (Càng cao càng tốt)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('PSNR (dB)')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

# 3. So sánh SSIM
for i, (name, res) in enumerate(SCREENING_RESULTS.items()):
    c = colors[i % len(colors)]
    axes[2].plot(range(1, len(res['history']['val_ssim']) + 1), res['history']['val_ssim'], 
                 label=name, color=c, marker='^', linewidth=2)
axes[2].set_title('So Sánh Validation SSIM (Càng gần 1.0 càng tốt)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('SSIM')
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend()

plt.tight_layout()
plt.show()

### Trực Quan Hóa Kết Quả Của Mô Hình Tốt Nhất (Best Model Demo)

In [ ]:
best_checkpoint_path = SCREENING_RESULTS[best_exp_name]['checkpoint']
print(f" Đang tải mô hình tốt nhất từ: {best_checkpoint_path}")

best_model = MoEQFormerDeblurNet(
    img_channel=3, 
    out_channels=3,
    width=COMMON_CONFIG['width'],
    enc_blk_nums=COMMON_CONFIG['enc_blk_nums'],
    middle_blk_num=COMMON_CONFIG['middle_blk_num'],
    dec_blk_nums=COMMON_CONFIG['dec_blk_nums'],
    num_queries=COMMON_CONFIG['num_queries'],
    num_experts=COMMON_CONFIG['num_experts']
).to(device)

if best_checkpoint_path != 'NAFNet-GoPro-width64.pth' and os.path.exists(best_checkpoint_path):
    best_model.load_state_dict(torch.load(best_checkpoint_path, map_location=device))
best_model.eval()

indices = random.sample(range(len(test_ds)), min(3, len(test_ds)))

for idx in indices:
    blur_tensor, sharp_tensor = test_ds[idx]
    blur_input = blur_tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        deblur_tensor, _ = best_model(blur_input)
        deblur_tensor = torch.clamp(deblur_tensor, 0.0, 1.0).squeeze(0).cpu()
        
    score_psnr = calc_psnr(deblur_tensor, sharp_tensor)
    score_ssim = calc_ssim(deblur_tensor, sharp_tensor)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(blur_tensor.permute(1, 2, 0).numpy())
    axes[0].set_title("Input: Ảnh Mờ (Blur)", fontsize=12)
    axes[0].axis("off")
    
    axes[1].imshow(deblur_tensor.permute(1, 2, 0).numpy())
    axes[1].set_title(f"Output: {best_exp_name}\nPSNR: {score_psnr:.2f} dB | SSIM: {score_ssim:.4f}", 
                      fontsize=12, color='darkgreen', fontweight='bold')
    axes[1].axis("off")
    
    axes[2].imshow(sharp_tensor.permute(1, 2, 0).numpy())
    axes[2].set_title("Target: Ảnh Gốc Nét (Sharp)", fontsize=12)
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()